In [1]:
import tensorflow as tf 
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [2]:
tf.test.is_gpu_available()

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


True

In [3]:
!pip install keras-tuner==1.1.0


   ---------------------------------------- 0.0/98.0 kB ? eta -:--:--
   ------------ --------------------------- 30.7/98.0 kB 1.3 MB/s eta 0:00:01
   ------------------------- -------------- 61.4/98.0 kB 656.4 kB/s eta 0:00:01
   ---------------------------------------- 98.0/98.0 kB 797.9 kB/s eta 0:00:00
   ---------------------------------------- 0.0/152.9 kB ? eta -:--:--
   -------------------------------- ------- 122.9/152.9 kB 3.6 MB/s eta 0:00:01
   ---------------------------------------- 152.9/152.9 kB 1.8 MB/s eta 0:00:00
  Attempting uninstall: cachetools
    Found existing installation: cachetools 5.3.3
    Uninstalling cachetools-5.3.3:
      Successfully uninstalled cachetools-5.3.3
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.22.0
    Uninstalling google-auth-2.22.0:
      Successfully uninstalled google-auth-2.22.0
  Attempting uninstall: keras-tuner
    Found existing installation: keras-tuner 1.4.7
    Uninstalling keras-tuner-1.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.6.0 requires clang~=5.0, which is not installed.
tensorflow 2.6.0 requires absl-py~=0.10, but you have absl-py 2.1.0 which is incompatible.
tensorflow 2.6.0 requires flatbuffers~=1.12, but you have flatbuffers 20210226132247 which is incompatible.


In [4]:
# Cell 2: Import Libraries
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras_tuner import BayesianOptimization
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt


In [6]:

train_dataset = 'brain-tumor-mri-dataset\Training'  
val_dataset = 'brain-tumor-mri-dataset\Testing'

train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dataset,  
    target_size=(224, 224),
    batch_size=32,
    class_mode='sparse'
)

val_generator = val_datagen.flow_from_directory(
    val_dataset,  
    target_size=(224, 224),
    batch_size=32,
    class_mode='sparse'
)

Found 5712 images belonging to 4 classes.
Found 1311 images belonging to 4 classes.


In [8]:
# Cell 4: Define Hypermodel
def build_model(hp):
    model = keras.Sequential()

    # Convolutional layers with tunable number of filters
    model.add(keras.layers.Conv2D(
        filters=hp.Int('filters_1', min_value=32, max_value=128, step=32),
        kernel_size=(3, 3),
        activation='relu',
        input_shape=(128, 128, 3)))
    model.add(keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(keras.layers.Conv2D(
        filters=hp.Int('filters_2', min_value=64, max_value=256, step=32),
        kernel_size=(3, 3),
        activation='relu'))
    model.add(keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(keras.layers.Conv2D(
        filters=hp.Int('filters_3', min_value=128, max_value=512, step=64),
        kernel_size=(3, 3),
        activation='relu'))
    model.add(keras.layers.MaxPooling2D(pool_size=(2, 2)))

    # Flatten before feeding into LSTM
    model.add(keras.layers.Flatten())

    # Reshape and add an LSTM layer with tunable units
    model.add(keras.layers.Reshape((1, -1)))  # Reshape to (1, features) to feed into LSTM
    model.add(keras.layers.LSTM(hp.Int('lstm_units', min_value=32, max_value=128, step=32)))

    # Dropout for regularization with a tunable dropout rate
    model.add(keras.layers.Dropout(hp.Float('dropout', min_value=0.2, max_value=0.5, step=0.1)))

    # Dense layer with softmax activation for multiclass classification
    model.add(keras.layers.Dense(num_classes, activation='softmax'))

    # Compile the model with a tunable learning rate
    model.compile(optimizer=keras.optimizers.Adam(
                      hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
                  loss='categorical_crossentropy',  # Multiclass loss function
                  metrics=['accuracy'])
    
    return model


In [9]:
# Cell 5: Initialize the Bayesian Tuner
from keras_tuner import BayesianOptimization

tuner = BayesianOptimization(
    build_model,
    objective='val_accuracy',  # Tune for validation accuracy
    max_trials=20,             # Number of hyperparameter combinations to try
    executions_per_trial=1,     # Number of models to train per trial
    directory='tuner_results',  # Where to save tuner logs
    project_name='brain_tumor_cnn_lstm_tuning')


ModuleNotFoundError: No module named 'tensorflow.python.trackable'